# Model Notebook

- Source: `src/model.py`
- 목적: 원본 파이썬 파일을 단계별로 실행/설명하기 위한 노트북 버전
- 실행 방법: 위에서 아래로 순서대로 실행


In [ ]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists():
    # 노트북이 다른 경로에서 열렸을 때 프로젝트 루트 자동 탐색
    for parent in [Path.cwd(), *Path.cwd().parents]:
        if (parent / 'src').exists() and (parent / 'configs').exists():
            PROJECT_ROOT = parent
            break
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print(f'Project root: {PROJECT_ROOT}')


## Step 1. Setup and Imports

이 셀은 원본 코드의 해당 블록을 그대로 옮긴 단계입니다.


In [ ]:
"""Single-head classifier (timm backbone + MLP head).

구조 요약
─────────
  입력 이미지 (3 × H × W)
       ↓
  timm backbone (예: efficientnet_b0)
  → global average pooling → feature vector (1280-dim for B0)
       ↓
  MLP head: Linear → ReLU → Dropout → Linear
       ↓
  logits (클래스 수 = n_classes)

왜 head를 따로 붙이나?
- timm으로 불러온 모델의 마지막 분류층(fc)은 ImageNet 1000-class용이다.
- num_classes=0 옵션으로 그 층을 제거하고, 우리 태스크(4 or 17 클래스)에
  맞는 head를 새로 붙인다.
- warmup 단계에서 backbone을 동결(freeze)하고 head만 먼저 학습할 수 있다.
"""
from __future__ import annotations
import torch.nn as nn
import timm


## Step 2. Class: CoffeeClassifier

이 셀은 원본 코드의 해당 블록을 그대로 옮긴 단계입니다.


In [ ]:
class CoffeeClassifier(nn.Module):


## Step 3. Function: __init__

이 셀은 원본 코드의 해당 블록을 그대로 옮긴 단계입니다.


In [ ]:
    def __init__(
        self,
        backbone: str = "efficientnet_b0",
        n_classes: int = 4,
        pretrained: bool = True,
        dropout: float = 0.3,
        hidden: int = 256,
    ):
        super().__init__()
        # num_classes=0 으로 만들면 timm 모델의 마지막 분류층을 제거하고
        # feature extractor(backbone)처럼 사용할 수 있다.
        self.backbone = timm.create_model(
            backbone, pretrained=pretrained, num_classes=0, global_pool="avg"
        )
        feat = self.backbone.num_features
        # backbone이 뽑은 feature를 한 번 더 비선형 변환한 뒤
        # 최종 클래스 수에 맞는 logits를 만든다.
        self.head = nn.Sequential(
            nn.Linear(feat, hidden),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(hidden, n_classes),
        )


## Step 4. Function: forward

이 셀은 원본 코드의 해당 블록을 그대로 옮긴 단계입니다.


In [ ]:
    def forward(self, x):
        # 흐름: image tensor → backbone feature → classification head → logits
        # backbone은 (B, feat) shape의 feature를 반환하고,
        # head가 이를 받아 (B, n_classes) logits를 만든다.
        return self.head(self.backbone(x))


## Step 5. Function: freeze_backbone

이 셀은 원본 코드의 해당 블록을 그대로 옮긴 단계입니다.


In [ ]:
    def freeze_backbone(self, freeze: bool = True) -> None:
        """backbone 파라미터의 gradient 계산을 켜거나 끈다.

        사용 시점
        - freeze=True  : warm-up epoch 동안 backbone을 고정, head만 학습
        - freeze=False : warm-up 끝난 뒤 backbone까지 함께 미세조정(fine-tune)

        이렇게 하는 이유: pretrained backbone은 이미 좋은 feature를 갖고 있다.
        처음부터 전체를 학습하면 head의 랜덤 초기값이 backbone을 망가뜨릴 수 있어서
        head를 먼저 안정시킨 뒤 backbone을 함께 학습하는 2-stage 방식을 쓴다.
        """
        for p in self.backbone.parameters():
            p.requires_grad = not freeze
